<div style="background:linear-gradient(135deg,#0D0F1A,#151828);border:1px solid #1E2340;border-radius:12px;padding:28px 36px;font-family:'Segoe UI',sans-serif;">
<h1 style="color:#7B2FFF;margin:0 0 4px;">🧠 Cobalt Transformer — Architecture</h1>
<p style="color:#9AADCC;margin:0;">Visual dissection of the CobaltTransformer: token flow, attention masks, positional encoding, and FFN dynamics.</p>
</div>

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import cobalt_utils as cu

cu.apply_cobalt_theme()
cfg  = cu.load_config()
arch = cfg["architecture"]
d, n_h, n_l, d_ff, seq = arch["d_model"], arch["n_heads"], arch["n_layers"], arch["d_ff"], arch["max_seq_len"]
print(f"✅  Config loaded — d={d}, heads={n_h}, layers={n_l}, d_ff={d_ff}, seq={seq}")

## 1 · Causal Attention Mask
The model uses an upper-triangular mask to enforce autoregressive (left-to-right) generation.

In [ ]:
cu.apply_cobalt_theme()

S = 12   # Display 12×12 for clarity
mask = np.triu(np.ones((S, S)), k=1)   # 1 = masked out (cannot attend)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Raw mask ───────────────────────────────────────────────────────────────
ax = axes[0]
cmap = plt.cm.get_cmap("Blues_r").copy()
im = ax.imshow(1 - mask, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_title(f"Causal Mask ({S}×{S} prefix)\nGreen = can attend | Red = masked")
ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
plt.colorbar(im, ax=ax, fraction=0.046)

# ── Attention pattern simulation ──────────────────────────────────────────
ax = axes[1]
np.random.seed(42)
attn_logits = np.random.randn(S, S)
attn_logits[mask.astype(bool)] = -1e9

# Softmax per row
def softmax_rows(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

attn_weights = softmax_rows(attn_logits)
im2 = ax.imshow(attn_weights, cmap="viridis", aspect="auto")
ax.set_title("Simulated Attention Weights (post-softmax)")
ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
plt.colorbar(im2, ax=ax, fraction=0.046)

plt.suptitle("CobaltTransformer — Causal Self-Attention", fontsize=15, color=cu.COBALT_CYAN)
plt.tight_layout()
plt.show()

## 2 · Positional Encoding Heatmap
Sinusoidal positional embeddings give each token a unique position signature.

In [ ]:
cu.apply_cobalt_theme()

# Learnable pos embeddings are random-init in Burn — we simulate sinusoidal for visualisation
positions = np.arange(seq)
dims      = np.arange(d)
PE        = np.zeros((seq, d))
for pos in positions:
    for i in range(0, d, 2):
        PE[pos, i]   = np.sin(pos / (10000 ** (i / d)))
        if i + 1 < d:
            PE[pos, i+1] = np.cos(pos / (10000 ** (i / d)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
im = ax.imshow(PE, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
ax.set_title(f"Positional Encoding Heatmap\n({seq} positions × {d} dims)")
ax.set_xlabel("Embedding Dimension"); ax.set_ylabel("Sequence Position")
plt.colorbar(im, ax=ax, fraction=0.046)

ax = axes[1]
for i in [0, 10, 30, 60]:
    ax.plot(PE[i, :40], label=f"pos={i}", linewidth=1.8)
ax.set_title("PE Slice — First 40 Dims at Selected Positions")
ax.set_xlabel("Dimension"); ax.set_ylabel("Encoding Value")
ax.legend(fontsize=8)

plt.suptitle("Positional Encoding Visualisation", fontsize=15, color=cu.COBALT_CYAN)
plt.tight_layout()
plt.show()

## 3 · Token Flow Through Transformer Blocks
Simulate how a random embedding evolves through `n_layers` blocks.

In [ ]:
cu.apply_cobalt_theme()
np.random.seed(7)

# Simulate residual stream norms across blocks (random walk + noise)
n_tokens = 8
x = np.random.randn(n_tokens, d) * 0.02  # small init

norms = [np.linalg.norm(x, axis=1).copy()]
for _ in range(n_l):
    # Simulate attn residual + ffn residual
    x = x + np.random.randn(*x.shape) * 0.15
    x = x + np.tanh(x @ np.random.randn(d, d_ff) * 0.01) @ np.random.randn(d_ff, d) * 0.01
    norms.append(np.linalg.norm(x, axis=1).copy())

norms = np.array(norms)  # shape: (n_layers+1, n_tokens)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Norm heatmap ───────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(norms.T, cmap="plasma", aspect="auto")
ax.set_xticks(range(n_l + 1))
ax.set_xticklabels(["Input"] + [f"Block {i+1}" for i in range(n_l)])
ax.set_yticks(range(n_tokens))
ax.set_yticklabels([f"Token {i}" for i in range(n_tokens)])
ax.set_title("Residual Stream L2 Norm (simulated)")
plt.colorbar(im, ax=ax, fraction=0.046)

# ── Per-token norm lines ───────────────────────────────────────────────────
ax = axes[1]
colours = plt.cm.plasma(np.linspace(0.2, 0.9, n_tokens))
for i in range(n_tokens):
    ax.plot(norms[:, i], color=colours[i], marker="o", linewidth=1.8, label=f"T{i}")
ax.set_xticks(range(n_l + 1))
ax.set_xticklabels(["Input"] + [f"Block {i+1}" for i in range(n_l)])
ax.set_title("Residual Norm per Token across Blocks")
ax.set_ylabel("L2 Norm"); ax.legend(fontsize=8, ncol=2)

plt.suptitle("Token Flow — Residual Stream Dynamics", fontsize=15, color=cu.COBALT_PURPLE)
plt.tight_layout()
plt.show()

## 4 · FFN Activation (GELU) vs ReLU

In [ ]:
cu.apply_cobalt_theme()

x_range = np.linspace(-4, 4, 300)
gelu  = x_range * 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (x_range + 0.044715 * x_range**3)))
relu  = np.maximum(0, x_range)
swish = x_range / (1 + np.exp(-x_range))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_range, gelu,  color=cu.COBALT_CYAN,   linewidth=2.5, label="GELU (used in Cobalt)")
ax.plot(x_range, relu,  color=cu.COBALT_BLUE,   linewidth=1.8, linestyle="--", label="ReLU")
ax.plot(x_range, swish, color=cu.COBALT_PURPLE, linewidth=1.8, linestyle=":",  label="Swish")
ax.axhline(0, color=cu.COBALT_GRID, linewidth=0.8)
ax.axvline(0, color=cu.COBALT_GRID, linewidth=0.8)
ax.set_title("FFN Activation Functions — GELU vs Alternatives", fontsize=14, pad=12)
ax.set_xlabel("Input"); ax.set_ylabel("Output")
ax.legend()
plt.tight_layout()
plt.show()

print("🧠 Cobalt uses GELU — smoother gradients than ReLU → better convergence at small scale.")